# Data_Preprocessing Solutions

Solutions for `exercises.ipynb`. Try the exercises first — peek here only after you've attempted each question.

## Part 1 — Warm-up

**1. Health check first.** Three problems hide in eight rows: holes (`NaN`), a text column, and — later — incomparable scales.

In [ ]:
import numpy as np
import pandas as pd

df = pd.DataFrame({
    "city":        ["Dhaka", "Sylhet", None, "Dhaka",
                    "Chattogram", "Sylhet", None, "Dhaka"],
    "age":         [25.0, 32.0, np.nan, 41.0, 28.0, np.nan, 35.0, 23.0],
    "income_taka": [52000.0, np.nan, 61000.0, 48000.0,
                    np.nan, 72000.0, 55000.0, 40000.0],
    "purchased":   [1, 0, 1, 0, 1, 0, 0, 1],
})

print(df)
print("\nMissing values per column:")
print(df.isna().sum())

**2. Fill the numeric holes.** The imputer learns each column's mean into `statistics_`, then pours that mean into every hole — no rows lost.

In [ ]:
from sklearn.impute import SimpleImputer

num_cols = ["age", "income_taka"]
print("Before:", df[num_cols].isna().sum().to_dict())

imp = SimpleImputer(strategy="mean")
filled = imp.fit_transform(df[num_cols])       # learns means, fills holes
num_filled = pd.DataFrame(filled, columns=num_cols)

print("After :", num_filled.isna().sum().to_dict())
print("Means learned:", dict(zip(num_cols, imp.statistics_.round(1))))

**3. Name the unknowns.** A constant fill keeps every row AND preserves missingness as its own learnable category.

In [ ]:
from sklearn.impute import SimpleImputer

imp_cat = SimpleImputer(strategy="constant", fill_value="Unknown")
city_fixed = imp_cat.fit_transform(df[["city"]]).ravel()

print(pd.Series(city_fixed).value_counts())

# Dropping rows throws away data; an explicit "Unknown" keeps them and lets
# the model learn whether hidden-city customers behave differently.

## Part 2 — Practice

**4. Sizes have an order.** Without `categories=` the encoder sorts alphabetically (L < M < S < XL) — nonsense the model would happily believe.

In [ ]:
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder

sizes = pd.DataFrame({"tshirt": ["M", "S", "XL", "L", "M", "S"]})

sizes["alphabetical"] = OrdinalEncoder().fit_transform(
    sizes[["tshirt"]]).ravel().astype(int)
sizes["meaningful_order"] = OrdinalEncoder(
    categories=[["S", "M", "L", "XL"]]).fit_transform(
    sizes[["tshirt"]]).ravel().astype(int)

print(sizes.to_string(index=False))

# Alphabetical encoding claims L(0) < M(2) < S(3) < XL(4) - clothing chaos.
# When a real ranking exists, spell it out via categories=[...].

**5. Cities have no order.** Each row lights exactly one 1 — membership declared with zero fake arithmetic between categories.

In [ ]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder

clean_city = df[["city"]].fillna("Unknown")

ohe = OneHotEncoder(sparse_output=False)
dummies = ohe.fit_transform(clean_city)

one_hot = pd.DataFrame(dummies.astype(int),
                       columns=ohe.get_feature_names_out(["city"]))
print(pd.concat([clean_city.reset_index(drop=True), one_hot], axis=1))
print("\nOnes per row:", one_hot.sum(axis=1).tolist())

# City codes like Sylhet(2) > Dhaka(1) invent order AND distance; KNN would
# treat city 2 as "twice as far" from city 0. One-hot states membership
# without any of those lies.

**6. Two rulers.** StandardScaler lands on mean 0 / std 1; MinMaxScaler squeezes into [0, 1]. Same shape, new units.

In [ ]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

standard = pd.DataFrame(StandardScaler().fit_transform(num_filled),
                        columns=num_cols)
minmax = pd.DataFrame(MinMaxScaler().fit_transform(num_filled),
                      columns=num_cols)

print("StandardScaler ->")
print(standard.describe().loc[["mean", "std", "min", "max"]].round(2))
print("\nMinMaxScaler ->")
print(minmax.describe().loc[["mean", "std", "min", "max"]].round(2))

# Standard: centred at 0 with spread 1.  MinMax: strictly inside [0, 1].
# Identical information - only the ruler changed.

**7. Leakage detective.** Only the train-fitted scaler's constants are legitimate; the other one smuggled test-row statistics into preprocessing.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = num_filled
y = df["purchased"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25,
                                          random_state=42)

leaky = StandardScaler().fit(X)         # peeked at EVERYTHING
honest = StandardScaler().fit(X_tr)     # saw training rows only

print("leaky-scaled test means :", leaky.transform(X_te).mean(axis=0).round(3))
print("honest-scaled test means:",
      honest.transform(X_te).mean(axis=0).round(3))

# The second line is clean: its constants came from train rows, so nothing
# from the "exam" leaked backwards. Iron rule: FIT ON TRAIN, TRANSFORM
# EVERYWHERE ELSE.

## Part 3 — Challenge

**8. One-stop preprocessing.** One fitted object imputes, scales and one-hots in a single call — and politely maps unseen future categories to all-zeros.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocess = ColumnTransformer([
    ("nums", make_pipeline(SimpleImputer(strategy="mean"), StandardScaler()),
     ["age", "income_taka"]),
    ("cats", make_pipeline(
        SimpleImputer(strategy="constant", fill_value="Unknown"),
        OneHotEncoder(handle_unknown="ignore")),
     ["city"]),
])

X_ready = preprocess.fit_transform(df.drop(columns="purchased"))

print("Output columns:", preprocess.get_feature_names_out().tolist())
print("Shape:", X_ready.shape, "- all numeric, zero NaN.")

# handle_unknown="ignore": tomorrow's customer from an untrained city maps
# to all-zeros instead of crashing the serving pipeline.

**9. Scaling pays the rent.** Unscaled, KNN picked neighbours by the loudest column (income noise); fixing the ruler recovered the real signal — free accuracy.

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(42)
n = 240
age = rng.uniform(18, 70, n)
income = np.clip(rng.normal(60000, 25000, n), 8000, None)
y_big = (age < 42).astype(int)
X_big = pd.DataFrame({"age": age, "income_taka": income})

Xb_tr, Xb_te, yb_tr, yb_te = train_test_split(
    X_big, y_big, test_size=0.25, random_state=42, stratify=y_big)

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(Xb_tr, yb_tr)
before = accuracy_score(yb_te, knn.predict(Xb_te))

scaler = StandardScaler().fit(Xb_tr)               # train rows only!
knn.fit(scaler.transform(Xb_tr), yb_tr)
after = accuracy_score(yb_te, knn.predict(scaler.transform(Xb_te)))

print(f"KNN accuracy WITHOUT scaling: {before:.2f}")
print(f"KNN accuracy WITH    scaling: {after:.2f}")

# Distance models sum comparisons across columns: income (tens of thousands)
# shouted over age (tens), so neighbours were chosen by noise. Correcting
# units - no new data, no tuning - hands the accuracy back for free.